# Small and Specialized Models: When Not to Use a Frontier LLM

The full code of [Small and Specialized Models: When Not to Use a Frontier LLM](https://residual-stream.lvtan71.workers.dev/posts/small-and-specialized-models/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/small-and-specialized-models) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/small-and-specialized-models.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 14 min end to end on the RTX 3090 machine that saved the outputs below.
routing.py times the encoder on the CPU and the language model on the GPU if there is one, so on a CUDA machine the speed ratio can invert.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 14 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.17.*" huggingface_hub "sentence-transformers==6.1.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "small-and-specialized-models"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/small-and-specialized-models


## `bandwidth.py`

The bandwidth bound on batch-1 decoding: weight sizes of five models in 16 and 4 bits, the most tokens per second a laptop and an H100 could reach, and a check against Qwen2.5-1.5B's measured 41 ms per token.

In [2]:
!{sys.executable} bandwidth.py

           BF16     INT4   tokens/s at most, batch 1:  laptop BF16 / INT4    H100 BF16 / INT4
0.5B      1.0GB   0.25GB                                  93 /   372     3350 / 13400
1.5B      3.0GB   0.75GB                                  31 /   124     1117 /  4467
3B        6.0GB   1.50GB                                  16 /    62      558 /  2233
8B       16.0GB   4.00GB                                   - /    23      209 /   838
70B     140.0GB  35.00GB                                   - /     -        - /    96

Qwen2.5-1.5B in 16 bits: 3.08 GB per token read; at the measured 41 ms per token that's 75 GB/s, 81% of the measured bandwidth


The post printed:

```text
           BF16     INT4   tokens/s at most, batch 1:  laptop BF16 / INT4    H100 BF16 / INT4
0.5B      1.0GB   0.25GB                                  93 /   372     3350 / 13400
1.5B      3.0GB   0.75GB                                  31 /   124     1117 /  4467
3B        6.0GB   1.50GB                                  16 /    62      558 /  2233
8B       16.0GB   4.00GB                                   - /    23      209 /   838
70B     140.0GB  35.00GB                                   - /     -        - /    96

Qwen2.5-1.5B in 16 bits: 3.08 GB per token read; at the measured 41 ms per token that's 75 GB/s, 81% of the measured bandwidth
```

## `routing.py`

Four-way routing of the series' 200 English router questions, one out-of-sample prediction each: the bag-of-words router, MiniLM sentence embeddings with a softmax head and by nearest centroid, and Qwen2.5-1.5B-Instruct zero-shot and with 20 examples, read typed and free. Accuracy with bootstrap intervals, calibration, batch-1 time.

In [3]:
!{sys.executable} routing.py

Qwen2.5-1.5B zero-shot: prompts of 118 tokens on average; the route spellings carry 99% of the first-word probability (median); free answers unparseable: 0 of 200


Qwen2.5-1.5B, 20 examples: prompts of 439 tokens on average; the route spellings carry 98% of the first-word probability (median); free answers unparseable: 0 of 200



                                                 accuracy    ECE  batch-1 time
bag-of-words router                      0.655 [0.59, 0.71]  0.050       15.5 us
MiniLM + softmax head                    0.710 [0.65, 0.77]  0.035       86.7 ms
MiniLM, nearest centroid                 0.780 [0.72, 0.83]  0.158       86.7 ms
Qwen2.5-1.5B zero-shot, typed            0.540 [0.47, 0.61]  0.291       23.0 ms
Qwen2.5-1.5B zero-shot, free             0.525 [0.46, 0.59]    n/a       47.8 ms
Qwen2.5-1.5B, 20 examples, typed         0.565 [0.50, 0.64]  0.275       32.1 ms
Qwen2.5-1.5B, 20 examples, free          0.550 [0.48, 0.62]    n/a       76.4 ms



MiniLM head minus Qwen with examples (typed): +0.145 [+0.06, +0.23]
MiniLM head, answering only at confidence >= 0.6: 70% of questions answered, 82.7% of those right
MiniLM head, answering only at confidence >= 0.7: 57% of questions answered, 88.6% of those right
MiniLM head, answering only at confidence >= 0.8: 41% of questions answered, 92.7% of those right
Qwen with examples, typed, right by true route: serving 0.92, architecture 0.96, agents 0.12, foundations 0.26


The post printed:

```text
Qwen2.5-1.5B zero-shot: prompts of 118 tokens on average; the route spellings carry 99% of the first-word probability (median); free answers unparseable: 0 of 200
Qwen2.5-1.5B, 20 examples: prompts of 439 tokens on average; the route spellings carry 98% of the first-word probability (median); free answers unparseable: 0 of 200

                                                 accuracy    ECE  batch-1 time
bag-of-words router                      0.655 [0.59, 0.71]  0.050        6.0 us
MiniLM + softmax head                    0.710 [0.65, 0.77]  0.035        2.6 ms
MiniLM, nearest centroid                 0.780 [0.72, 0.83]  0.158        2.6 ms
Qwen2.5-1.5B zero-shot, typed            0.540 [0.47, 0.61]  0.291      155.9 ms
Qwen2.5-1.5B zero-shot, free             0.525 [0.46, 0.59]    n/a      215.8 ms
Qwen2.5-1.5B, 20 examples, typed         0.565 [0.50, 0.64]  0.274      465.6 ms
Qwen2.5-1.5B, 20 examples, free          0.555 [0.49, 0.62]    n/a      521.4 ms

MiniLM head minus Qwen with examples (typed): +0.145 [+0.06, +0.23]
MiniLM head, answering only at confidence >= 0.6: 70% of questions answered, 82.7% of those right
MiniLM head, answering only at confidence >= 0.7: 57% of questions answered, 88.6% of those right
MiniLM head, answering only at confidence >= 0.8: 41% of questions answered, 92.7% of those right
Qwen with examples, typed, right by true route: serving 0.92, architecture 0.96, agents 0.12, foundations 0.26
```